# Regresión Lineal Simple y Múltiple

_Supuestos, interpretación, métricas y criterios de información_

**Módulo 1 — Aprendizaje Supervisado** | DSRP Machine Learning Engineering

![Aprendizaje Supervisado](assets/header.png)

## 1. Regresión lineal simple

Modelamos la relación entre una variable objetivo continua $y$ y un único predictor $x$ mediante una recta:

$$
y_i = \beta_0 + \beta_1 x_i + \varepsilon_i
$$

donde $\beta_0$ es el **intercepto**, $\beta_1$ la **pendiente** y $\varepsilon_i$ es el error aleatorio. La idea visual es buscar la recta que **mejor pasa por los puntos**: la que minimiza la suma de los cuadrados de las distancias verticales (residuos).

Los parámetros $\hat{\beta}_0, \hat{\beta}_1$ se calculan con una fórmula cerrada:

$$
\hat{\beta}_1 = \frac{\sum_i (x_i - \bar{x})(y_i - \bar{y})}{\sum_i (x_i - \bar{x})^2}
\qquad
\hat{\beta}_0 = \bar{y} - \hat{\beta}_1 \bar{x}
$$

In [ ]:
# --- Visualización de la idea de mínimos cuadrados ---
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
xd = np.linspace(0, 10, 25)
yd = 2 + 0.8 * xd + rng.normal(0, 1.2, len(xd))
b1 = np.cov(xd, yd, ddof=0)[0, 1] / np.var(xd)
b0 = yd.mean() - b1 * xd.mean()
yhat = b0 + b1 * xd

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(xd, yd, color='steelblue', label='datos')
ax.plot(xd, yhat, color='red', lw=2, label='recta OLS')
for x, y, yh in zip(xd, yd, yhat):
    ax.plot([x, x], [y, yh], color='gray', alpha=0.5, lw=1)
ax.set_title('Mínimos cuadrados: minimizar la suma de residuos²')
ax.legend(); ax.grid(True); plt.show()

## 2. Regresión lineal múltiple

Generalizamos a $p$ predictores:

$$
y_i = \beta_0 + \beta_1 x_{i1} + \beta_2 x_{i2} + \dots + \beta_p x_{ip} + \varepsilon_i
$$

Geométricamente ya no es una recta sino un **hiperplano** en el espacio de los predictores. La idea es la misma: encontrar los $\beta$ que minimicen la suma de cuadrados de los residuos.

## 3. Supuestos del modelo lineal

1. **Linealidad** — la relación entre $\mathbb{E}[y]$ y los predictores es lineal en los $\beta$.
2. **Independencia** de los errores.
3. **Homocedasticidad** — varianza constante de los errores (no embudo en los residuos).
4. **Normalidad** de los errores (necesaria para los p-valores e intervalos de confianza).
5. **No multicolinealidad** entre predictores. Se vigila con el **VIF** (Variance Inflation Factor): valores $> 5$–$10$ son señal de alerta.

## 4. Interpretación de coeficientes

Cada $\hat{\beta}_j$ se interpreta como **el cambio esperado en $y$ ante un incremento de una unidad en $x_j$, manteniendo el resto constante** (ceteris paribus).

El test $t$ del summary contrasta $H_0: \beta_j = 0$. Un **p-valor** pequeño (< 0.05) sugiere que la variable aporta información.

## 5. Métricas de desempeño

$$
\text{MAE} = \frac{1}{n}\sum |y_i - \hat{y}_i| \qquad
\text{RMSE} = \sqrt{\frac{1}{n}\sum (y_i - \hat{y}_i)^2}
$$

$$
R^2 = 1 - \frac{SS_{res}}{SS_{tot}}
\qquad
R^2_{adj} = 1 - (1 - R^2)\,\frac{n - 1}{n - p - 1}
$$

- **MAE / RMSE**: error en las mismas unidades que $y$. RMSE penaliza más los errores grandes.
- **R²**: proporción de la varianza de $y$ explicada por el modelo (entre 0 y 1).
- **R² ajustado**: penaliza agregar variables que no aportan, útil para comparar modelos.

## 6. Criterios de información (AIC, BIC)

Sirven para **comparar modelos**: penalizan la complejidad. Sea $\hat{L}$ la verosimilitud del modelo, $k$ el número de parámetros y $n$ el tamaño muestral:

$$
\text{AIC} = 2k - 2\ln \hat{L}
\qquad
\text{BIC} = k \ln(n) - 2\ln \hat{L}
$$

**Menor es mejor**. BIC penaliza más fuerte la complejidad cuando $n$ es grande, así que tiende a elegir modelos más simples que AIC.

## 7. Caso práctico — House Prices (Kaggle)

**Dataset:** https://www.kaggle.com/c/house-prices-advanced-regression-techniques

**Estructura.** Es una competencia de Kaggle, así que viene partida en dos archivos:

| Archivo | Filas | Columnas | `SalePrice` |
|---|---|---|---|
| `housing_train.csv` | 1.460 | 81 | ✅ presente |
| `housing_test.csv`  | 1.459 | 80 | ❌ ausente (se usa para enviar al leaderboard) |

Cada fila es una vivienda en Ames, Iowa, con 79 atributos descriptivos (calidad, área habitable, año de construcción, sótano, garage, baños, vecindario, etc.) más un `Id`. Como el `test.csv` no trae la etiqueta, **para entrenar y evaluar usamos solo `housing_train.csv`** y lo partimos nosotros con `train_test_split`.

**Problema:** regresión sobre `SalePrice` (precio de venta en USD).

**Subconjunto de variables que usaremos** (numéricas y de interpretación clara):
`GrLivArea` (área habitable sobre el suelo, ft²), `OverallQual` (calidad general 1–10), `YearBuilt` (año de construcción), `GarageCars` (capacidad del garage en autos), `TotalBsmtSF` (área total del sótano, ft²), `FullBath` (baños completos).

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path('../data/housing_train.csv')
if not DATA.exists():
    raise FileNotFoundError(
        f'No se encontró {DATA}. Descarga housing_train.csv desde '
        'https://www.kaggle.com/c/house-prices-advanced-regression-techniques '
        'y colócalo en data/ (ver README.md).'
    )

df = pd.read_csv(DATA)
print('Shape:', df.shape)
df.head()

In [ ]:
import seaborn as sns
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid')

features = ['GrLivArea', 'OverallQual', 'YearBuilt', 'GarageCars',
            'TotalBsmtSF', 'FullBath']
data = df[features + ['SalePrice']].dropna()
print('Observaciones tras limpiar nulos:', len(data))
data.describe().round(2)

In [ ]:
# --- Regresión SIMPLE: SalePrice ~ GrLivArea ---
X = data[['GrLivArea']]
y = data['SalePrice']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
lr = LinearRegression().fit(X_tr, y_tr)

print(f'Intercepto β0 = {lr.intercept_:,.2f}')
print(f'Pendiente  β1 = {lr.coef_[0]:,.2f}  (USD por pie² adicional)')

y_pred = lr.predict(X_te)
print(f'MAE   = {mean_absolute_error(y_te, y_pred):,.0f}')
print(f'RMSE  = {np.sqrt(mean_squared_error(y_te, y_pred)):,.0f}')
print(f'R²    = {r2_score(y_te, y_pred):.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.scatterplot(x='GrLivArea', y='SalePrice', data=data, alpha=0.4, ax=ax)
xs = np.linspace(data['GrLivArea'].min(), data['GrLivArea'].max(), 100).reshape(-1, 1)
ax.plot(xs, lr.predict(xs), color='red', lw=2, label='OLS')
ax.set_title('Regresión simple: SalePrice ~ GrLivArea')
ax.legend()
plt.show()

In [ ]:
# --- Regresión MÚLTIPLE con statsmodels (inferencia) ---
X = data[features]
y = data['SalePrice']

X_sm = sm.add_constant(X)
ols = sm.OLS(y, X_sm).fit()
print(ols.summary())

In [ ]:
print(f'AIC          = {ols.aic:,.1f}')
print(f'BIC          = {ols.bic:,.1f}')
print(f'R²           = {ols.rsquared:.4f}')
print(f'R² ajustado  = {ols.rsquared_adj:.4f}')

In [ ]:
# --- Diagnóstico: residuos vs ajustados (homocedasticidad) ---
fitted = ols.fittedvalues
resid = ols.resid

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(fitted, resid, alpha=0.4)
axes[0].axhline(0, color='red', ls='--')
axes[0].set_xlabel('Valores ajustados'); axes[0].set_ylabel('Residuos')
axes[0].set_title('Residuos vs ajustados')

sm.qqplot(resid, line='45', fit=True, ax=axes[1])
axes[1].set_title('Q-Q plot de residuos')
plt.tight_layout(); plt.show()

In [ ]:
# --- VIF para detectar multicolinealidad ---
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif = pd.DataFrame({
    'feature': X.columns,
    'VIF': [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
})
vif

## 8. Referencias

- ISLR cap. 3: *Linear Regression*. https://www.statlearning.com/
- Akaike, H. (1974). *A new look at the statistical model identification*.
- Schwarz, G. (1978). *Estimating the dimension of a model* (BIC).
- Documentación de statsmodels: https://www.statsmodels.org/stable/regression.html
- Dataset: https://www.kaggle.com/c/house-prices-advanced-regression-techniques